<a href="https://colab.research.google.com/github/Dhanshree010/pattern-recognition-practical/blob/main/practical1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from scipy.spatial.distance import euclidean, cityblock, cosine, hamming
import warnings
warnings.filterwarnings("ignore")


In [2]:
print("=" * 60)
print("  STEP 1: Load and Explore Dataset")
print("=" * 60)

df = pd.read_csv("diabetes.csv")

  STEP 1: Load and Explore Dataset


In [3]:
print(f"\nDataset Shape : {df.shape}")
print(f"Features      : {list(df.columns)}")
print(f"\nFirst 5 rows:\n{df.head()}")
print(f"\nBasic Statistics:\n{df.describe().round(2)}")



Dataset Shape : (768, 9)
Features      : ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome']

First 5 rows:
   Pregnancies  Glucose  BloodPressure  SkinThickness  Insulin   BMI  \
0            6      148             72             35        0  33.6   
1            1       85             66             29        0  26.6   
2            8      183             64              0        0  23.3   
3            1       89             66             23       94  28.1   
4            0      137             40             35      168  43.1   

   DiabetesPedigreeFunction  Age  Outcome  
0                     0.627   50        1  
1                     0.351   31        0  
2                     0.672   32        1  
3                     0.167   21        0  
4                     2.288   33        1  

Basic Statistics:
       Pregnancies  Glucose  BloodPressure  SkinThickness  Insulin     BMI  \
count       768.00   768

In [4]:
print("\n" + "=" * 60)
print("  STEP 2: Pattern Representation (Feature Vector)")
print("=" * 60)

# Drop target column — we use only features for similarity
features = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness',
            'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age']

X = df[features].copy()

# Normalize features to [0, 1] so distances are not scale-biased
scaler = MinMaxScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=features)

print("\nOriginal feature vector for Patient 0:")
print(X.iloc[0].to_dict())

print("\nNormalized feature vector for Patient 0 (scale 0–1):")
print({k: round(v, 4) for k, v in X_scaled.iloc[0].to_dict().items()})

print("\nEach row (patient) is a point in 8-dimensional feature space.")



  STEP 2: Pattern Representation (Feature Vector)

Original feature vector for Patient 0:
{'Pregnancies': 6.0, 'Glucose': 148.0, 'BloodPressure': 72.0, 'SkinThickness': 35.0, 'Insulin': 0.0, 'BMI': 33.6, 'DiabetesPedigreeFunction': 0.627, 'Age': 50.0}

Normalized feature vector for Patient 0 (scale 0–1):
{'Pregnancies': 0.3529, 'Glucose': 0.7437, 'BloodPressure': 0.5902, 'SkinThickness': 0.3535, 'Insulin': 0.0, 'BMI': 0.5007, 'DiabetesPedigreeFunction': 0.2344, 'Age': 0.4833}

Each row (patient) is a point in 8-dimensional feature space.


In [5]:
print("\n" + "=" * 60)
print("  STEP 3: Select 5 Sample Customer Profiles")
print("=" * 60)

sample_indices = [0, 1, 2, 3, 4]
samples = X_scaled.iloc[sample_indices]

print("\nNormalized profiles for selected customers:")
print(samples.round(4).to_string())



  STEP 3: Select 5 Sample Customer Profiles

Normalized profiles for selected customers:
   Pregnancies  Glucose  BloodPressure  SkinThickness  Insulin     BMI  DiabetesPedigreeFunction     Age
0       0.3529   0.7437         0.5902         0.3535   0.0000  0.5007                    0.2344  0.4833
1       0.0588   0.4271         0.5410         0.2929   0.0000  0.3964                    0.1166  0.1667
2       0.4706   0.9196         0.5246         0.0000   0.0000  0.3472                    0.2536  0.1833
3       0.0588   0.4472         0.5410         0.2323   0.1111  0.4188                    0.0380  0.0000
4       0.0000   0.6884         0.3279         0.3535   0.1986  0.6423                    0.9436  0.2000


In [6]:
print("\n" + "=" * 60)
print("  STEP 4: Proximity Measure 1 — Euclidean Distance")
print("=" * 60)
print("""
Formula:  d(A, B) = sqrt( Σ (Aᵢ - Bᵢ)² )
Interpretation: Straight-line distance in feature space.
                SMALLER value → MORE similar customers.
""")

n = len(sample_indices)
euc_matrix = np.zeros((n, n))

for i in range(n):
    for j in range(n):
        euc_matrix[i][j] = euclidean(samples.iloc[i], samples.iloc[j])

euc_df = pd.DataFrame(euc_matrix,
                       index=[f"C{i}" for i in sample_indices],
                       columns=[f"C{i}" for i in sample_indices])

print("Euclidean Distance Matrix:")
print(euc_df.round(4).to_string())

# Find most similar pair (ignore diagonal)
np.fill_diagonal(euc_matrix, np.inf)
min_idx = np.unravel_index(np.argmin(euc_matrix), euc_matrix.shape)
print(f"\n→ Most similar pair : C{sample_indices[min_idx[0]]} "
      f"& C{sample_indices[min_idx[1]]} "
      f"(distance = {euc_matrix[min_idx]:.4f})")


  STEP 4: Proximity Measure 1 — Euclidean Distance

Formula:  d(A, B) = sqrt( Σ (Aᵢ - Bᵢ)² )
Interpretation: Straight-line distance in feature space.
                SMALLER value → MORE similar customers.

Euclidean Distance Matrix:
        C0      C1      C2      C3      C4
C0  0.0000  0.5638  0.5367  0.6948  0.9161
C1  0.5638  0.0000  0.7209  0.2256  0.9518
C2  0.5367  0.7209  0.0000  0.7379  1.0205
C3  0.6948  0.2256  0.7379  0.0000  1.0196
C4  0.9161  0.9518  1.0205  1.0196  0.0000

→ Most similar pair : C1 & C3 (distance = 0.2256)


In [7]:
print("\n" + "=" * 60)
print("  STEP 5: Proximity Measure 2 — Manhattan Distance")
print("=" * 60)
print("""
Formula:  d(A, B) = Σ |Aᵢ - Bᵢ|
Interpretation: Sum of absolute differences along each axis.
                Also called L1 or City-Block distance.
                SMALLER value → MORE similar customers.
""")

man_matrix = np.zeros((n, n))

for i in range(n):
    for j in range(n):
        man_matrix[i][j] = cityblock(samples.iloc[i], samples.iloc[j])

man_df = pd.DataFrame(man_matrix,
                       index=[f"C{i}" for i in sample_indices],
                       columns=[f"C{i}" for i in sample_indices])

print("Manhattan Distance Matrix:")
print(man_df.round(4).to_string())

np.fill_diagonal(man_matrix, np.inf)
min_idx = np.unravel_index(np.argmin(man_matrix), man_matrix.shape)
print(f"\n→ Most similar pair : C{sample_indices[min_idx[0]]} "
      f"& C{sample_indices[min_idx[1]]} "
      f"(distance = {man_matrix[min_idx]:.4f})")


  STEP 5: Proximity Measure 2 — Manhattan Distance

Formula:  d(A, B) = Σ |Aᵢ - Bᵢ|
Interpretation: Sum of absolute differences along each axis.
                Also called L1 or City-Block distance.
                SMALLER value → MORE similar customers.

Manhattan Distance Matrix:
        C0      C1      C2      C3      C4
C0  0.0000  1.2593  1.1854  1.6338  2.0032
C1  1.2593  0.0000  1.4165  0.4594  1.8987
C2  1.1854  1.4165  0.0000  1.7145  2.4523
C3  1.6338  0.4594  1.7145  0.0000  2.0510
C4  2.0032  1.8987  2.4523  2.0510  0.0000

→ Most similar pair : C1 & C3 (distance = 0.4594)


In [8]:
print("\n" + "=" * 60)
print("  STEP 6: Proximity Measure 3 — Cosine Similarity")
print("=" * 60)
print("""
Formula:  cos(A, B) = (A · B) / (||A|| × ||B||)
          similarity = 1 - cosine_distance
Interpretation: Measures angle between two vectors.
                Ignores magnitude; captures direction/pattern.
                HIGHER value (closer to 1) → MORE similar.
""")

cos_matrix = np.zeros((n, n))

for i in range(n):
    for j in range(n):
        # scipy.cosine gives distance (0=identical, 2=opposite)
        # similarity = 1 - distance
        dist = cosine(samples.iloc[i] + 1e-10, samples.iloc[j] + 1e-10)
        cos_matrix[i][j] = round(1 - dist, 6)

cos_df = pd.DataFrame(cos_matrix,
                       index=[f"C{i}" for i in sample_indices],
                       columns=[f"C{i}" for i in sample_indices])

print("Cosine Similarity Matrix:")
print(cos_df.round(4).to_string())

np.fill_diagonal(cos_matrix, -np.inf)
max_idx = np.unravel_index(np.argmax(cos_matrix), cos_matrix.shape)
print(f"\n→ Most similar pair : C{sample_indices[max_idx[0]]} "
      f"& C{sample_indices[max_idx[1]]} "
      f"(similarity = {cos_matrix[max_idx]:.4f})")



  STEP 6: Proximity Measure 3 — Cosine Similarity

Formula:  cos(A, B) = (A · B) / (||A|| × ||B||)
          similarity = 1 - cosine_distance
Interpretation: Measures angle between two vectors.
                Ignores magnitude; captures direction/pattern.
                HIGHER value (closer to 1) → MORE similar.

Cosine Similarity Matrix:
        C0      C1      C2      C3      C4
C0  1.0000  0.9402  0.9121  0.8708  0.7823
C1  0.9402  1.0000  0.8266  0.9663  0.7707
C2  0.9121  0.8266  1.0000  0.8172  0.7222
C3  0.8708  0.9663  0.8172  1.0000  0.7197
C4  0.7823  0.7707  0.7222  0.7197  1.0000

→ Most similar pair : C1 & C3 (similarity = 0.9663)


In [9]:
print("\n" + "=" * 60)
print("  STEP 7: Proximity Measure 4 — Edit Distance")
print("=" * 60)
print("""
Formula:  Minimum single-character edits (insert, delete, replace)
          needed to turn string A into string B.
Encoding: Each patient's profile is encoded as a categorical
          string (e.g., age group, glucose level, BMI category).
          SMALLER value → MORE similar profiles.
""")

# ── Encode continuous features into categorical bins ──────────
def encode_profile(row):
    """Convert a patient row into a compact category string."""
    age_cat    = "Y" if row["Age"] <= 30 else ("M" if row["Age"] <= 50 else "O")
    glucose    = "L" if row["Glucose"] < 100 else ("N" if row["Glucose"] < 140 else "H")
    bp         = "L" if row["BloodPressure"] < 70 else ("N" if row["BloodPressure"] < 90 else "H")
    bmi        = "U" if row["BMI"] < 18.5 else ("N" if row["BMI"] < 25 else
                 ("P" if row["BMI"] < 30 else "O"))
    insulin    = "L" if row["Insulin"] < 50 else ("N" if row["Insulin"] < 150 else "H")
    preg       = "Z" if row["Pregnancies"] == 0 else ("L" if row["Pregnancies"] < 3 else "H")
    return age_cat + glucose + bp + bmi + insulin + preg

# Use the original (un-normalised) values for encoding
df_sample = df.iloc[sample_indices].reset_index(drop=True)
profiles  = [encode_profile(df_sample.iloc[i]) for i in range(n)]

print("Encoded String Profiles:")
for idx, (sid, prof) in enumerate(zip(sample_indices, profiles)):
    print(f"  C{sid}: {prof}  ← {dict(df_sample.iloc[idx][['Age','Glucose','BloodPressure','BMI','Insulin','Pregnancies']])}")

# ── Wagner-Fischer dynamic programming edit distance ──────────
def edit_distance(s1, s2):
    """Classic DP Levenshtein / Edit Distance."""
    m, n = len(s1), len(s2)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m + 1):
        dp[i][0] = i
    for j in range(n + 1):
        dp[0][j] = j
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if s1[i - 1] == s2[j - 1]:
                dp[i][j] = dp[i - 1][j - 1]
            else:
                dp[i][j] = 1 + min(dp[i - 1][j],    # delete
                                   dp[i][j - 1],    # insert
                                   dp[i - 1][j - 1]) # replace
    return dp[m][n]

edit_matrix = np.zeros((n, n), dtype=int)
for i in range(n):
    for j in range(n):
        edit_matrix[i][j] = edit_distance(profiles[i], profiles[j])

edit_df = pd.DataFrame(edit_matrix,
                        index=[f"C{i}" for i in sample_indices],
                        columns=[f"C{i}" for i in sample_indices])

print("\nEdit Distance Matrix:")
print(edit_df.to_string())

tmp = edit_matrix.copy().astype(float)
np.fill_diagonal(tmp, np.inf)
min_idx = np.unravel_index(np.argmin(tmp), tmp.shape)
print(f"\n→ Most similar pair : C{sample_indices[min_idx[0]]} "
      f"& C{sample_indices[min_idx[1]]} "
      f"(edit distance = {edit_matrix[min_idx]})")



  STEP 7: Proximity Measure 4 — Edit Distance

Formula:  Minimum single-character edits (insert, delete, replace)
          needed to turn string A into string B.
Encoding: Each patient's profile is encoded as a categorical
          string (e.g., age group, glucose level, BMI category).
          SMALLER value → MORE similar profiles.

Encoded String Profiles:
  C0: MHNOLH  ← {'Age': np.float64(50.0), 'Glucose': np.float64(148.0), 'BloodPressure': np.float64(72.0), 'BMI': np.float64(33.6), 'Insulin': np.float64(0.0), 'Pregnancies': np.float64(6.0)}
  C1: MLLPLL  ← {'Age': np.float64(31.0), 'Glucose': np.float64(85.0), 'BloodPressure': np.float64(66.0), 'BMI': np.float64(26.6), 'Insulin': np.float64(0.0), 'Pregnancies': np.float64(1.0)}
  C2: MHLNLH  ← {'Age': np.float64(32.0), 'Glucose': np.float64(183.0), 'BloodPressure': np.float64(64.0), 'BMI': np.float64(23.3), 'Insulin': np.float64(0.0), 'Pregnancies': np.float64(8.0)}
  C3: YLLPNL  ← {'Age': np.float64(21.0), 'Glucose': np.floa

In [10]:
print("\n" + "=" * 60)
print("  STEP 8: Comparison Summary — All Measures")
print("=" * 60)

pairs = [(i, j) for i in range(n) for j in range(i + 1, n)]

rows = []
for (i, j) in pairs:
    ci, cj = sample_indices[i], sample_indices[j]
    euc  = euclidean(samples.iloc[i], samples.iloc[j])
    man  = cityblock(samples.iloc[i], samples.iloc[j])
    cos  = 1 - cosine(samples.iloc[i] + 1e-10, samples.iloc[j] + 1e-10)
    ed   = edit_distance(profiles[i], profiles[j])
    rows.append({
        "Pair"            : f"C{ci}-C{cj}",
        "Euclidean ↓"     : round(euc, 4),
        "Manhattan ↓"     : round(man, 4),
        "Cosine Sim ↑"    : round(cos, 4),
        "Edit Dist ↓"     : ed
    })

summary_df = pd.DataFrame(rows)
print(summary_df.to_string(index=False))

print("""
Notes:
  ↓  = lower is more similar
  ↑  = higher is more similar
""")



  STEP 8: Comparison Summary — All Measures
 Pair  Euclidean ↓  Manhattan ↓  Cosine Sim ↑  Edit Dist ↓
C0-C1       0.5638       1.2593        0.9402            4
C0-C2       0.5367       1.1854        0.9121            2
C0-C3       0.6948       1.6338        0.8708            6
C0-C4       0.9161       2.0032        0.7823            4
C1-C2       0.7209       1.4165        0.8266            3
C1-C3       0.2256       0.4594        0.9663            2
C1-C4       0.9518       1.8987        0.7707            4
C2-C3       0.7379       1.7145        0.8172            4
C2-C4       1.0205       2.4523        0.7222            4
C3-C4       1.0196       2.0510        0.7197            5

Notes:
  ↓  = lower is more similar
  ↑  = higher is more similar



In [11]:
print("=" * 60)
print("  STEP 9: Top-3 Customers Most Similar to Customer C0")
print("         (using Euclidean Distance on full dataset)")
print("=" * 60)

query = X_scaled.iloc[0].values   # Customer 0 as the query profile

distances = []
for idx in range(1, len(X_scaled)):
    d = euclidean(query, X_scaled.iloc[idx].values)
    distances.append((idx, round(d, 4)))

distances.sort(key=lambda x: x[1])
top3 = distances[:3]

print(f"\nQuery: Customer C0  →  Profile: {df.iloc[0][features].to_dict()}\n")
print(f"{'Rank':<6} {'Customer ID':<14} {'Euclidean Dist':<18} {'Profile (original values)'}")
print("-" * 80)
for rank, (cid, dist) in enumerate(top3, 1):
    profile = df.iloc[cid][features].to_dict()
    print(f"{rank:<6} C{cid:<13} {dist:<18} {profile}")

print("\n→ These are the most similar customers — ideal for collaborative filtering!")


  STEP 9: Top-3 Customers Most Similar to Customer C0
         (using Euclidean Distance on full dataset)

Query: Customer C0  →  Profile: {'Pregnancies': 6.0, 'Glucose': 148.0, 'BloodPressure': 72.0, 'SkinThickness': 35.0, 'Insulin': 0.0, 'BMI': 33.6, 'DiabetesPedigreeFunction': 0.627, 'Age': 50.0}

Rank   Customer ID    Euclidean Dist     Profile (original values)
--------------------------------------------------------------------------------
1      C701           0.1624             {'Pregnancies': 6.0, 'Glucose': 125.0, 'BloodPressure': 78.0, 'SkinThickness': 31.0, 'Insulin': 0.0, 'BMI': 27.6, 'DiabetesPedigreeFunction': 0.565, 'Age': 49.0}
2      C754           0.1775             {'Pregnancies': 8.0, 'Glucose': 154.0, 'BloodPressure': 78.0, 'SkinThickness': 32.0, 'Insulin': 0.0, 'BMI': 32.4, 'DiabetesPedigreeFunction': 0.443, 'Age': 45.0}
3      C603           0.194              {'Pregnancies': 7.0, 'Glucose': 150.0, 'BloodPressure': 78.0, 'SkinThickness': 29.0, 'Insulin': 126.0, 

In [12]:
print("\n" + "=" * 60)
print("  CONCLUSION")
print("=" * 60)
print("""
Measure           | Best For
──────────────────┼─────────────────────────────────────────────
Euclidean         | Continuous features, geometric closeness
Manhattan         | Robust to outliers, grid-like feature spaces
Cosine Similarity | Pattern/direction similarity, sparse data
Edit Distance     | Categorical or sequence-based profiles

All four measures successfully identify similar customer (patient)
profiles. In a real e-commerce system:
  • Euclidean / Manhattan → collaborative filtering on purchase history
  • Cosine Similarity     → content-based recommendation (TF-IDF vectors)
  • Edit Distance         → matching user behaviour sequences or categories
""")



  CONCLUSION

Measure           | Best For
──────────────────┼─────────────────────────────────────────────
Euclidean         | Continuous features, geometric closeness
Manhattan         | Robust to outliers, grid-like feature spaces
Cosine Similarity | Pattern/direction similarity, sparse data
Edit Distance     | Categorical or sequence-based profiles
 
All four measures successfully identify similar customer (patient)
profiles. In a real e-commerce system:
  • Euclidean / Manhattan → collaborative filtering on purchase history
  • Cosine Similarity     → content-based recommendation (TF-IDF vectors)
  • Edit Distance         → matching user behaviour sequences or categories

